>[REGEX](#updateTitle=true&folderId=1yOparXp7vN8_wLhCUg2G2rs7EW7KTWFO&scrollTo=qIOeCKA9dKCr)

>[NLP toolbox](#updateTitle=true&folderId=1yOparXp7vN8_wLhCUg2G2rs7EW7KTWFO&scrollTo=R9FU627Ct6yk)



# 1. REGEX

Patterns: Dates, IDs / codes, Medications + doses, Vital signs.
https://www.johnsnowlabs.com/extract-medical-named-entities-with-regex-in-healthcare-nlp-at-scale

In [ ]:
#Libraries
import re
import pandas as pd

In [ ]:
#Extract meds, dosages, dates, vitals
notes = [
    """Pt ID: 12345. 2025-03-01: Pt seen in clinic.
    Hx of HTN, type 2 diabetes. BP 150/95, HR 88.
    Started metformin 500 mg BID and lisinopril 10 mg daily.
    ICD-10: I21.9, E11.9. Denies chest pain.""",

    """Patient: 67890, visit 03/15/2025.
    Complains of chest pain on exertion. BP 130/85, HR 72.
    ICD10 code I20.0 was assigned. Prescribed atorvastatin 20 mg qhs.""",

    """ID: 11111. 2025-02-20: routine follow-up.
    No chest pain, no shortness of breath. BP 118/76, HR 65.
    Continue metformin 1000 mg BID."""
]

pat_pattern = re.compile(r"\b([A-Za-z]{1,8}\s?[A-Za-z]{0,2})\s*:\s*(\d{5}\b)")
date_pattern_iso = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
date_pattern_us = re.compile(r"\b\d{1,2}/\d{1,2}/\d{4}\b")
bp_pattern = re.compile(r"\bBP\s*(\d{2,3})/(\d{2,3})\b", re.IGNORECASE)
hr_pattern = re.compile(r"\bHR\s*(\d{2,3})\b", re.IGNORECASE)
med_dose_pattern = re.compile(r"\b([A-Za-z]+)\s+(\d+)\s?(mg|mcg|ml)\b", re.IGNORECASE)

rows = []

for note in notes:
  pat_id_match = pat_pattern.search(note)
  dates = date_pattern_iso.findall(note) + date_pattern_us.findall(note)
  bp_match = bp_pattern.search(note)
  hr_match = hr_pattern.search(note)
  med_dose_match = med_dose_pattern.findall(note)

  row = {
      "row_note": note.strip(),
       "patient_id": pat_id_match.group(2) if pat_id_match else None,
      "dates": dates,
      "bp_systolic": int(bp_match.group(1)) if bp_match else None,
      "bp_diastolic": int(bp_match.group(2)) if bp_match else None,
      "heart_rate": int(hr_match.group(1)) if hr_match else None,
      "meds_raw": med_dose_match,
      "meds_structure": [
          {"drug": m[0].lower(),
              "dose": int(m[1]),
              "unit": m[2].lower()}
              for m in med_dose_match
      ]
  }

  rows.append(row)

df = pd.DataFrame(rows)
print(df[["patient_id", "dates", "bp_systolic", "bp_diastolic", "heart_rate", "meds_structure"]])

  patient_id         dates  bp_systolic  bp_diastolic  heart_rate  \
0      12345  [2025-03-01]          150            95          88   
1      67890  [03/15/2025]          130            85          72   
2      11111  [2025-02-20]          118            76          65   

                                      meds_structure  
0  [{'drug': 'metformin', 'dose': 500, 'unit': 'm...  
1  [{'drug': 'atorvastatin', 'dose': 20, 'unit': ...  
2  [{'drug': 'metformin', 'dose': 1000, 'unit': '...  


ICD-10 codes, “Denies chest pain” vs “reports chest pain”.

In [ ]:
#ICD-10-ish pattern
icd10_pattern = re.compile(r"\b[A-TV-Z][0-9]{2}(?:\.[0-9A-Z]{1,4})?\b")

#Negated chest pain
neg_chest_pain_pattern = re.compile(
    r"(denies|no|without)\s+chest pain",
    re.IGNORECASE
)

#Any chest pain mention
chest_pain_pattern = re.compile(r"chest pain", re.IGNORECASE)

rows = []

for note in notes:
    dates = date_pattern_iso.findall(note) + date_pattern_us.findall(note)
    bp_match = bp_pattern.search(note)
    hr_match = hr_pattern.search(note)
    med_dose_match = med_dose_pattern.findall(note)
    icd_codes = icd10_pattern.findall(note)

    #negation vs positive
    neg_mention = bool(neg_chest_pain_pattern.search(note))
    any_mention = bool(chest_pain_pattern.search(note))

    chest_pain_status = "no_mention"
    if any_mention and not neg_mention:
        chest_pain_status = "reported"
    elif any_mention and neg_mention:
        chest_pain_status = "negated"

    row = {
        "raw_note": note.strip(),
        "dates": dates,
        "bp_systolic": int(bp_match.group(1)) if bp_match else None,
        "bp_diastolic": int(bp_match.group(2)) if bp_match else None,
        "heart_rate": int(hr_match.group(1)) if hr_match else None,
        "meds_structured": [
            {"drug": m[0].lower(), "dose": int(m[1]), "unit": m[2].lower()}
            for m in med_dose_match
        ],
        "icd10_codes": icd_codes,
        "chest_pain_status": chest_pain_status,
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(df[["icd10_codes", "chest_pain_status"]])


      icd10_codes chest_pain_status
0  [I21.9, E11.9]           negated
1         [I20.0]          reported
2              []           negated


# 2. NLP toolbox

Preprocessing (tokenization, lemmatization, stopwords); classical features(TF-IDF with a linear model); spaCy / scispaCy / medspaCy for clinical-ish NER.

scispaCy for biomedical/clinical text.
medspaCy toolkit built on spaCy for clinical notes.

https://spacy.io/usage/models \
https://allenai.github.io/scispacy/


Language: en (English)
Type: core (Vocabulary, syntax, entities)
Genre: web (written text (blogs, news, comments))
Size: sm (12 MB)

## 2.1 Preprocessing

In [ ]:
#Install
#%%capture
# %pip install -q "spacy>=3.7.0,<3.8.0" "scispacy>=0.5.5"
# %pip install -q "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz"
# %pip install -q "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz"
# import spacy, scispacy
# print("spaCy:", spacy.__version__)
# print("scispaCy:", scispacy.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 MB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
spaCy: 3.7.5
scispaCy: 0.6.2


In [ ]:
#Libraries
import spacy
from spacy.matcher import Matcher, PhraseMatcher
from spacy import displacy

In [ ]:
#Small English model trained on web text
nlp = spacy.load("en_core_web_sm")

#"en_ner_bc5cdr_md" recognizes drugs, diseases

/usr/local/lib/python3.12/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [ ]:
text = """
2025-03-01: Patient with type 2 diabetes and hypertension.
Started metformin 500 mg BID and lisinopril 10 mg daily.
Denies chest pain, but reports occasional shortness of breath.
"""

#Processes the entire text and returns a Doc object
#It contains tokens, sentences, lemmas, POS tags, dependencies, entitie, etc
doc = nlp(text)

#Splits the text into sentences
print("Sentences:")
for sent in doc.sents:
    print("-", sent.text.strip())

#.text original word
#.lemma_ base form (e.g. "Started" → "start")
#.pos_ part of speech (NOUN, VERB, NUM, etc.)

#Loops through every token (word/punctuation)
print("Tokens/lemmas:")
for token in doc:
   print(token.text, "->", token.lemma_, token.pos_)


print("Tokens, lemmas, POS:")
for token in doc:
    print(f"{token.text:<12} {token.lemma_:<12} {token.pos_:<6} {token.dep_:<10} head={token.head.text}")

#Named entities recognized by the model
print("Named entities:")
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_}")

print("Noun chunks:")
for chunk in doc.noun_chunks:
    print(f"{chunk.text:<30} -> head: {chunk.root.text}, label: {chunk.root.dep_}")

#Dependency parse tree of the first sentence
#Shows grammatical structure: who is subject, what is object, modifiers, etc
spacy.displacy.render(next(doc.sents), style='dep', jupyter=True)

Sentences:
- 2025-03-01: Patient with type 2 diabetes and hypertension.
- Started metformin 500 mg BID and lisinopril 10 mg daily.
- Denies chest pain, but reports occasional shortness of breath.
Tokens/lemmas:

 -> 
 SPACE
2025 -> 2025 NUM
- -> - PUNCT
03 -> 03 NUM
- -> - SYM
01 -> 01 NUM
: -> : PUNCT
Patient -> patient NOUN
with -> with ADP
type -> type NOUN
2 -> 2 NUM
diabetes -> diabete NOUN
and -> and CCONJ
hypertension -> hypertension NOUN
. -> . PUNCT

 -> 
 SPACE
Started -> started AUX
metformin -> metformin NOUN
500 -> 500 NUM
mg -> mg PROPN
BID -> BID PROPN
and -> and CCONJ
lisinopril -> lisinopril PROPN
10 -> 10 NUM
mg -> mg PROPN
daily -> daily PROPN
. -> . PUNCT

 -> 
 SPACE
Denies -> deny VERB
chest -> chest NOUN
pain -> pain NOUN
, -> , PUNCT
but -> but CCONJ
reports -> report VERB
occasional -> occasional ADJ
shortness -> shortness NOUN
of -> of ADP
breath -> breath NOUN
. -> . PUNCT

 -> 
 SPACE
Tokens, lemmas, POS:

            
            SPACE  dep        head=2025

In [ ]:
#Add rule based patterns
matcher = Matcher(nlp.vocab)

#Pattern: "chest pain" optionally preceded by adjective
chest_pain_pattern = [
    {"LOWER": {"IN": ["chest"]}}, #matches "chest", "Chest", "CHEST"
    {"LOWER": "pain"} #must be followed by "pain"
]

#Add pattern with a label "CHEST_PAIN"
matcher.add("CHEST_PAIN", [chest_pain_pattern])

#PhraseMatcher for specific drugs
phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
drug_terms = ["metformin", "lisinopril", "atorvastatin"]
drug_docs = [nlp.make_doc(t) for t in drug_terms]
phrase_matcher.add("DRUG", drug_docs)

#Add all drug terms under one label: "DRUG"
matches = matcher(doc)
print("\nMatcher: chest pain spans")
for match_id, start, end in matches:
    span = doc[start:end]
    print(span.text, "->", nlp.vocab.strings[match_id])

print("\nPhraseMatcher: drugs")
for match_id, start, end in phrase_matcher(doc):
    span = doc[start:end]
    print(span.text, "->", nlp.vocab.strings[match_id])


Matcher: chest pain spans
chest pain -> CHEST_PAIN

PhraseMatcher: drugs
metformin -> DRUG
lisinopril -> DRUG


In [ ]:
#Ppipeline for biomedical text (tokenizer, tagger, parser)
sci_nlp = spacy.load("en_core_sci_sm")

#NER model for diseases and chemicals/drugs
bc5cdr_nlp = spacy.load("en_ner_bionlp13cg_md")

text = """
The patient has poorly controlled type 2 diabetes and hypertension.
Metformin 500 mg BID and lisinopril 10 mg daily were started.
There is concern for diabetic neuropathy.
"""

sci_doc = sci_nlp(text)
bc_doc = bc5cdr_nlp(text)

print("SciSpaCy tokens and noun chunks:")
for chunk in sci_doc.noun_chunks:
    print(chunk.text, "->", chunk.root.text, chunk.root.dep_)

print("\nBC5CDR entities (diseases and chemicals):")
for ent in bc_doc.ents:
    print(f"{ent.text:<30} {ent.label_}")

/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


SciSpaCy tokens and noun chunks:
The patient -> patient nsubj
hypertension -> hypertension conj

Metformin 500 mg BID -> BID nsubjpass
lisinopril -> lisinopril conj
10 mg -> mg nsubjpass
concern -> concern nsubj

BC5CDR entities (diseases and chemicals):
patient                        ORGANISM
Metformin                      SIMPLE_CHEMICAL
BID                            GENE_OR_GENE_PRODUCT
lisinopril                     SIMPLE_CHEMICAL


## 2.2 TF-IDF, classifier on clinical-ish sentences

In [ ]:
%pip install -q "scikit-learn>=1.4.0" "gradio>=4.0.0" "openai>=1.0.0"

In [ ]:
#Libraries
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import gradio as gr
import os
import json
from openai import OpenAI
from google.colab import userdata

In [ ]:
#Configure OpenAI client
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

In [ ]:
#Classify notes as 'cardio' 'respiratory' 'diabetes'
texts = [
    "Chest pain, ST elevation on ECG, concern for acute MI.",
    "Long history of coronary artery disease and angina.",
    "Shortness of breath, orthopnea, likely heart failure.",
    "Fasting-glucose 180, HbA1c 9.5, poorly controlled diabetes.",
    "Metformin 1000 mg BID and insulin added to regimen.",
    "Peripheral neuropathy and diabetic retinopathy present.",
    "Wheezing, cough, diagnosed with asthma.",
    "Smoker with chronic cough, suspect COPD.",
    "Spirometry shows obstructive airway disease.",
    "Chest pain radiating to left arm, troponin elevated.",
    "New onset atrial fibrillation, started on apixaban.",
    "HbA1c 10.2, started on semaglutide.",
    "Chronic cough with sputum, diagnosed with COPD.",
    "Chest pain radiating to jaw, EKG changes.",
    "Patient with type 1 diabetes started on insulin pump.",
    "Severe asthma exacerbation, given nebulizer treatment.",
    "FEV1/FVC ratio 55%, diagnosed with COPD."
]

labels = [
    "cardio", "cardio", "cardio",
    "diabetes", "diabetes", "diabetes",
    "respiratory", "respiratory", "respiratory",
    "cardio", "cardio",
    "diabetes",
    "respiratory",
    "cardio", "diabetes", "respiratory", "respiratory"
]

#TF-IDF Term Frequency - Inverse Document Frequency
# "chest pain" - Very common in cardio, high score in cardio notes
# "troponin", "MI" - Almost only appear in heart attack notes, super strong signal
# "metformin", "HbA1c" - Almost only in diabetes notes, strong diabetes signal
# "patient", "with" - Appear everywhere, low TF-IDF score (not helpful)
clf = Pipeline([
    #ngram_range() looks at single words, pairs of words, triples
    #min_df includes a word/ngram even if it appears in only 1 document
    #lowercase converts everything to lowercase before processing,
    #"Patient" and "patient" are the same word
    #stop_words removes common words
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df = 1, lowercase=True, stop_words='english')),
    ("model", LogisticRegression(class_weight="balanced",
        solver="lbfgs"))
])

#Split data 6 training, 3 testing
#stratify=labels keep same proportion of each class in train/test
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

#ngram_range=(1,2) - looks at single words ()"chest", "pain", "metformin"),
#pairs: ()"chest pain", "coronary artery", "type 2")
#LogisticRegression() - if "chest pain", "troponin", "MI", predict cardio
#If "metformin", "HbA1c", "glucose", predict diabetes
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Classification report:")
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=["cardio", "diabetes", "respiratory"]))

#"chest pain", strong cardio signal
#"hypertension", also common in heart disease
#No diabetes words, confidently predicts cardio
test_text = "Patient with COPD and chronic wheezing, FEV1/FVC ratio 55% on spirometry."
print("\nPredicted class for test_text:", clf.predict([test_text]))

Classification report:
              precision    recall  f1-score   support

      cardio       0.50      0.50      0.50         2
    diabetes       0.67      1.00      0.80         2
 respiratory       1.00      0.50      0.67         2

    accuracy                           0.67         6
   macro avg       0.72      0.67      0.66         6
weighted avg       0.72      0.67      0.66         6

Confusion matrix:
[[1 1 0]
 [0 2 0]
 [1 0 1]]

Predicted class for test_text: ['respiratory']


# 3. LLM / OpenAI API

Call the API from Python; Summarize a note; Extract structured JSON: diagnoses, medications, follow-up; Run over a batch of notes.

In [ ]:
# pip install --upgrade openai
#!pip install -q gradio
%pip install -q gradio==4.44.0 openai==1.50.0 huggingface_hub==0.23.0
#!pip install gradio
#Libraries
import os
import json
from openai import OpenAI
from google.colab import userdata
import gradio as gr
import json

In [ ]:
#Pulls a secret API key from an environment
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

#Summarize the note into concise bullet points (using a natural language prompt)
def summarize_note(note_text: str) -> str:
    prompt = (
        "You are a clinical summarization assistant.\n\n"
        "Summarize the following clinical note in 3 bullet points, "
        "covering diagnosis, key treatments, and follow-up.\n\n"
        f"NOTE:\n{note_text}"
        )

    resp = client.responses.create(
        model = "gpt-5-nano",
        input = prompt
        )

    summary = resp.output_text
    return summary.strip() #Remove leading/trailing whitespace

In [ ]:
#Extracts key info as a structured JSON dict
def extract_structured(note_text: str) -> dict:
      prompt = (
        "Extract structured clinical information from the note below.\n"
        "Return ONLY valid JSON with keys: 'diagnosis', 'medications', 'follow_up'.\n"
        "'medications' should be a list of objects with keys 'name' and 'dose'.\n\n"
        f"NOTE:\n{note_text}"
        )

      resp = client.responses.create(
          model = "gpt-5-nano",
          input = prompt
          )

      raw_text = resp.output_text
      #Robust JSON parsing with fallback
      try:
          data = json.loads(raw_text)
      except json.JSONDecodeError:
          # Simple recovery: try to locate JSON substring
          try:
              start = raw_text.index("{")
              end = raw_text.rindex("}") + 1
              data = json.loads(raw_text[start:end])
          except Exception:
              raise ValueError(f"Model did not return valid JSON: {raw_text[:200]}...")

      return data

In [ ]:
note = """
2025-03-01: Patient with poorly controlled type 2 diabetes and hypertension.
BP 150/95, HbA1c 9.1. Start metformin 500 mg BID and increase lisinopril to 20 mg daily.
Follow-up in 3 months with labs.
"""

print("Summary:")
print(summarize_note(note))

print("\nStructured:")
print(extract_structured(note))

Summary:
- Diagnosis: Poorly controlled type 2 diabetes mellitus and hypertension (BP 150/95; HbA1c 9.1%).

- Key treatments: Initiation of metformin 500 mg twice daily; lisinopril increased to 20 mg daily.

- Follow-up: Reassess with labs in 3 months.

Structured:
{'diagnosis': ['Type 2 diabetes mellitus, poorly controlled', 'hypertension'], 'medications': [{'name': 'metformin', 'dose': '500 mg BID'}, {'name': 'lisinopril', 'dose': '20 mg daily'}], 'follow_up': '3 months with labs'}


In [ ]:
#Gradio interface for summarization and extraction
def summarize_and_extract(note_text):
    """Validate input, summarize note, and extract structured JSON."""

    #Input validation
    if not note_text or not note_text.strip():
        return "Please enter a clinical note.", "{}"

    if len(note_text) < 10:
        return "Note too short. Please enter a complete clinical note.", "{}"

    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OpenAI API key not configured.", "{}"

    try:
        summary = summarize_note(note_text)
        structured = extract_structured(note_text)

        # Format JSON nicely
        json_output = json.dumps(structured, indent=2, ensure_ascii=False)
        return summary, json_output

    except Exception as e:
        error_msg = f"Error processing note: {str(e)}"
        return error_msg, json.dumps({"error": str(e)}, indent=2)

#Interface
demo = gr.Interface(
    fn=summarize_and_extract,
    inputs=gr.Textbox(
        lines=15,
        label="Clinical Note",
        placeholder="Enter clinical note text here (avoid real PHI in demos)...",
        info="Paste a synthetic or de-identified clinical note to summarize and extract structured data"
    ),
    outputs=[
        gr.Textbox(label="Summary", lines=8),
        gr.Code(label="Structured Data (JSON)", language="json", lines=15)
    ],
    title="Clinical Note Summarizer",
    description="Extract structured information and generate summaries from clinical notes using AI",
    examples=[
        ["Pt ID: 12345. 2025-03-01: Pt seen in clinic. Hx of HTN, type 2 diabetes. BP 150/95, HR 88. Started metformin 500 mg BID."],
        ["Patient: 67890. Complains of chest pain. BP 130/85, HR 72. Prescribed atorvastatin 20 mg qhs."],
        ["Follow-up: Patient denies chest pain. BP 128/78. Continue lisinopril and metformin."]
    ],
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

demo.launch(share=True)
#Respiratory/asthma

#Try 2025-05-02: Patient with history of asthma presents with worsening wheeze and shortness of breath.
# Given nebulized albuterol and IV steroids in ED.
# Discharged with inhaled corticosteroid and follow-up in 4 weeks.

#Cardio admission

# 2025-04-10: 65-year-old with chest pain radiating to left arm and jaw.
# ECG with ST elevation in anterior leads, troponin elevated.
# Started on aspirin, ticagrelor, and heparin drip. Plan for urgent PCI.

# Diabetes + hypertension follow‑up

# 2025-03-01: Patient with poorly controlled type 2 diabetes and hypertension.
# BP 150/95, HbA1c 9.1. Start metformin 500 mg BID and increase lisinopril to 20 mg daily.
# Follow-up in 3 months with labs.

In [ ]:
#app.py with API key handling
app_code = '''import os
import json
import re
import gradio as gr
from openai import OpenAI
from typing import Tuple, Dict, Any

#Initialize client with retries
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    timeout=30.0,
    max_retries=3
)

#Robust JSON extraction for LLM text issues
def extract_json_from_text(text: str) -> Dict[str, Any]:
    # Try direct parse first
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass

    #Extract JSON substring
    json_match = re.search(r'\\{.*\\}', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except json.JSONDecodeError:
            pass

    return {
        "diagnosis": "Unable to extract",
        "medications": [],
        "follow_up": "Unable to extract"
    }


#Generate clinical summary with structured prompt
def summarize_note(note_text: str) -> str:
    messages = [
        {"role": "system", "content": "You are a clinical documentation specialist. Respond concisely."},
        {"role": "user", "content": f"""Summarize this clinical note in EXACTLY 3 bullet points:
- Diagnosis/Key findings
- Treatments/Medications started/changed
- Follow-up plan/next steps

CLINICAL NOTE:
{note_text}"""}
    ]

    resp = client.chat.completions.create(
        model="gpt-5-nano",  # Real, cost-effective model
        messages=messages,
        temperature=0.1,  # Low creativity for consistency
        max_tokens=200
    )

    return resp.choices[0].message.content.strip()

#Extract structured clinical data as JSON
def extract_structured(note_text: str) -> Dict[str, Any]:
    messages = [
        {"role": "system", "content": """You extract clinical data as JSON only. Never explain or add text outside JSON.
Output format: {{"diagnosis": "str", "medications": [{{"name": "str", "dose": "str"}}], "follow_up": "str"}}"""},
        {"role": "user", "content": f"NOTE: {note_text}"}
    ]

    resp = client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,
        temperature=0.0,  # Deterministic for JSON
        max_tokens=300
    )

    return extract_json_from_text(resp.choices[0].message.content)

#Main app function with full validation.
def summarize_and_extract(note_text: str) -> Tuple[str, str]:
    """Main app function with full validation."""
    # Input validation
    if not note_text or not note_text.strip():
        return "❌ Please enter a clinical note.", json.dumps({}, indent=2)

    if len(note_text.strip()) < 20:
        return "❌ Note too short (<20 chars). Enter a complete clinical note.", json.dumps({}, indent=2)

    # API key check
    if not os.environ.get("OPENAI_API_KEY"):
        return "❌ OpenAI API key missing. Add OPENAI_API_KEY in Space Settings.", json.dumps({}, indent=2)

    try:
        summary = summarize_note(note_text)
        structured = extract_structured(note_text)
        json_output = json.dumps(structured, indent=2, ensure_ascii=False)

        return summary, json_output

    except Exception as e:
        error_msg = f"Processing error: {str(e)}"
        return error_msg, json.dumps({"error": str(e)}, indent=2)

#Interface
demo = gr.Interface(
    fn=summarize_and_extract,
    inputs=gr.Textbox(
        lines=12,
        label="📋 Clinical Note",
        placeholder="Paste de-identified clinical note here (avoid real patient data)...",
        info="For demo only - do not use real PHI. Synthetic/test data recommended."
    ),
    outputs=[
        gr.Textbox(label="AI Summary", lines=6, max_lines=8),
        gr.Code(label="Structured JSON", language="json", lines=12)
    ],
    title="🏥 Clinical Note AI Assistant",
    description="""Summarize clinical notes and extract structured data (diagnosis, medications, follow-up).
**Demo only** - Uses external OpenAI API. Do not enter real patient data.""",
    examples=[
        ["2025-03-01: Pt w/ T2DM, HTN. BP 150/95, A1c 9.1%. Start metformin 500mg BID, ↑ lisinopril 20mg qd. F/U 3mo w/ labs."],
        ["65M c/o chest pain → L arm. ECG: anterior ST↑, troponin+. ASA 325, ticagrelor 180 load, heparin drip. Cath lab."],
        ["Asthma exacerbation. Nebs x3, IV solumedrol. D/c w/ symbicort 160/4.5 ii BID, f/u pulm 2wks."]
    ],
    theme=gr.themes.Soft(primary_hue="blue"),
    allow_flagging="never"
)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

In [ ]:
#Create requirements.txt
requirements = '''gradio>=4.44.0
openai>=1.50.0
huggingface_hub>=0.23.0
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements)

#Create README.md
readme = '''---
title: Clinical Note AI Assistant
emoji: 🏥
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 4.44.0
app_file: app.py
pinned: false
python_version: "3.10"  # FIXED: Explicit Python 3.10
---

# Clinical Note Summarizer & Extractor

AI-powered clinical documentation assistant using GPT-4o-mini.

## **DEMO ONLY** - Privacy Warning
- Uses external OpenAI API
- **Do NOT enter real patient data/PHI**
- Synthetic or de-identified notes only

## Setup
1. Add `OPENAI_API_KEY` secret in Space Settings
2. Wait for rebuild (1-2 min)

## Tested Versions
- Python 3.10 (HF Spaces default)
- Gradio 4.44.0
- OpenAI 1.50.0
- huggingface_hub 0.23.0
'''

with open('README.md', 'w') as f:
    f.write(readme)

#Login to Hugging Face
from huggingface_hub import notebook_login

print("Please login to Hugging Face:")
print("Get your token from: https://huggingface.co/settings/tokens")
print("Create a token with 'Write' access")

notebook_login()

Please login to Hugging Face:
Get your token from: https://huggingface.co/settings/tokens
Create a token with 'Write' access


In [ ]:
#Deploy to Hugging Face Spaces
from huggingface_hub import notebook_login, HfApi, create_repo
import os
import getpass

required_files = ['app.py', 'requirements.txt', 'README.md']
missing = [f for f in required_files if not os.path.exists(f)]

#Hugging Face username
SPACE_NAME = f"exprmcg/clinical-note-summarizer-ai"

api = HfApi()

#Create the Space
try:
    create_repo(
        repo_id=SPACE_NAME,
        repo_type="space",
        space_sdk="gradio",
        private=False,
        exist_ok=True  # NEW: skips if exists
    )
    print("Space created/verified")
except HfApiError as e:
    if "already exists" in str(e):
        print("Space already exists - proceeding to upload")
    else:
        raise

#Upload all files
files_to_upload = ['app.py', 'requirements.txt', 'README.md']
success_count = 0

for file_name in files_to_upload:
    if not os.path.exists(file_name):
        print(f"Skipping {file_name} (missing)")
        continue

    try:
        api.upload_file(
            path_or_fileobj=file_name,
            path_in_repo=file_name,
            repo_id=SPACE_NAME,
            repo_type="space"
        )
        print(f"Uploaded {file_name}")
        success_count += 1
    except Exception as e:
        if "already up to date" in str(e).lower():
            print(f"{file_name} already up to date")
        else:
            print(f"Upload failed {file_name}: {e}")
    except Exception as e:
        print(f"Unexpected error {file_name}: {e}")

print("COMPLETE!")

# 1.Go to the space: `https://huggingface.co/spaces/exprmcg/clinical-note-summarizer-ai`
# 2."Settings"
# 3."Variables and secrets"
# 4."New secret"
# 5. Add the secret:Name: OPENAI_API_KEY, Value: sk-proj-xxxxxxxxxxxxxxxxxxxxxxxx

Space created/verified
Uploaded app.py
Uploaded requirements.txt
Uploaded README.md
COMPLETE!

Wait 1-2 minutes for the Space to rebuild after adding the key


In [ ]:
# Test in Colab before deploying
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Test your functions
test_note = """
2025-03-01: Patient with poorly controlled type 2 diabetes and hypertension.
BP 150/95, HbA1c 9.1. Start metformin 500 mg BID and increase lisinopril to 20 mg daily.
Follow-up in 3 months with labs.
"""

summary, structured = summarize_and_extract(test_note)
print("Summary:", summary)
print("\nStructured:", structured)

# 4. MongoDB, SQL

Connecting using pymongo; Storing clinical notes as JSON documents; Querying by patient ID, specialty, date; Doing a basic aggregation (group by patient, count notes); Using a regex query on note_text; Simple table with id, patient_id, visit_date, note_text; SELECT with filters and simple aggregations.

In [ ]:
#Libraries
%%capture
from datetime import datetime
import re
import sqlite3
from datetime import date
import os
import time
import subprocess
!apt-get update -qq
!apt-get install -y docker.io -qq

#Start Docker service
!service docker start

#Run MongoDB in Docker
!docker run -d -p 27017:27017 --name mongodb mongo:4.4

#Wait for container to start
import time
time.sleep(10)

#!pip install pymongo

from pymongo import MongoClient

In [ ]:
#Connect to a local MongoDB database
client = MongoClient("mongodb://127.0.0.1:27017/")

# Test connection
try:
    client.server_info()
    print("Connected successfully!")
except Exception as e:
    print(f"Connection failed: {e}")

#Creates/accesses a database called clinical_db/ collection called notes
db = client["clinical_db"]
notes_col = db["notes"]

notes_col.delete_many({})  #Deletes all documents in the collection for testing

docs = [
    {
        "patient_id": "12345",
        "visit_date": datetime(2025, 3, 1),
        "specialty": "cardiology",
        "note_text": "2025-03-01: Chest pain, BP 150/95, started metoprolol 25 mg BID."
    },
    {
        "patient_id": "12345",
        "visit_date": datetime(2025, 6, 1),
        "specialty": "cardiology",
        "note_text": "Follow-up: improved symptoms, BP 130/80, continue metoprolol."
    },
    {
        "patient_id": "67890",
        "visit_date": datetime(2025, 3, 15),
        "specialty": "endocrinology",
        "note_text": "Type 2 diabetes, HbA1c 9.1, start metformin 500 mg BID."
    },
]

notes_col.insert_many(docs)

print("All cardiology notes:")
for doc in notes_col.find({"specialty": "cardiology"}):
    print(doc["patient_id"], doc["visit_date"], doc["note_text"])

print("\nNotes mentioning metformin (regex search):")
for doc in notes_col.find({"note_text": {"$regex": r"metformin", "$options": "i"}}):
    print(doc["patient_id"], doc["note_text"])

print("\nNumber of notes per patient (aggregation):")
pipeline = [
    {"$group": {"_id": "$patient_id", "note_count": {"$sum": 1}}},
    {"$sort": {"note_count": -1}}
]
for row in notes_col.aggregate(pipeline):
    print(row)


In [ ]:
#":memory:" creates a database entirely in RAM
#conn: connection to the database
#cur: cursor to run SQL commands
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

#Create a table with 5 columns
cur.execute("""
CREATE TABLE clinical_notes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id TEXT,
    visit_date DATE,
    specialty TEXT,
    note_text TEXT
)
""")

#List of sample data (as tuples)
rows = [
    ("12345", "2025-03-01", "cardiology", "Chest pain, BP 150/95, started metoprolol 25 mg BID."),
    ("12345", "2025-06-01", "cardiology", "Follow-up visit, BP 130/80, continue metoprolol."),
    ("67890", "2025-03-15", "endocrinology", "Type 2 diabetes, HbA1c 9.1, start metformin 500 mg BID.")
]

#executemany inserts all rows at once (fast & clean)
#? placeholders (prevents SQL injection — safe!)
#commit() saves the changes
cur.executemany("INSERT INTO clinical_notes (patient_id, visit_date, specialty, note_text) VALUES (?, ?, ?, ?)", rows)
conn.commit()

print("Cardiology notes:")
for row in cur.execute("SELECT patient_id, visit_date, note_text FROM clinical_notes WHERE specialty = 'cardiology'"):
    print(row)

print("\nNotes with 'diabetes':")
for row in cur.execute("SELECT patient_id, note_text FROM clinical_notes WHERE note_text LIKE '%diabetes%'"):
    print(row)

print("\nNote count per patient:")
for row in cur.execute("SELECT patient_id, COUNT(*) FROM clinical_notes GROUP BY patient_id"):
    print(row)


Cardiology notes:
('12345', '2025-03-01', 'Chest pain, BP 150/95, started metoprolol 25 mg BID.')
('12345', '2025-06-01', 'Follow-up visit, BP 130/80, continue metoprolol.')

Notes with 'diabetes':
('67890', 'Type 2 diabetes, HbA1c 9.1, start metformin 500 mg BID.')

Note count per patient:
('12345', 2)
('67890', 1)
